# Week 5 Lab 2: Advanced Regression & Pipelines

**Goal**: Predict medical charges using **Multiple Features** (Age, BMI, Smoking status) and **ColumnTransformer**.

> **Why this lab matters**:
> In the real world, models use hundreds of features of different types (numbers, text). You must know how to combine `StandardScaler` for numbers and `OneHotEncoder` for text into a single automated pipeline.

> **Structure**:
> We follow the **6-Phase Professional Workflow** but expand Phase 2 (Preprocessing) into parallel tracks using a `ColumnTransformer`. We will then use **Cross Validation** again to verify the massive accuracy boost.

---
## Foreword
We continue our work with the **Medical Insurance Dataset**. In Lab 1, we only used Age. Today, we unleash the model on more data.

1. **Phase 1: Splitting**
2. **Phase 2: Preprocessing (Parallel Transformation)**
3. **Phase 3: Assembly (Pipeline)**
4. **Phase 4: Training**
5. **Phase 5: Evaluation (R-Squared & Cross-Validation)**
6. **Phase 6: Optimization**

### 1.1 Import Dependencies & Load Data
**Concept**: We select multiple features. Notice that `smoker` is a categorical column ('yes' or 'no'), which AI cannot read algebraically.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

X = df[['age', 'bmi', 'smoker']]
y = df['charges']

---
### 1.2 Phase 1: Data Splitting
**Concept**: We secure 20% of the data for an unbiased final exam.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
#### Theory: Scikit-Learn ColumnTransformer
You cannot scale text, and you shouldn't One-Hot Encode numbers. `ColumnTransformer` routes different columns to different tools.

| Component | Target Columns | Function |
| :--- | :--- | :--- |
| `StandardScaler()` | `['age', 'bmi']` | Centers the numerical data around 0. |
| `OneHotEncoder()` | `['smoker']` | Converts 'yes'/'no' string into binary columns. |

### 1.3 Phase 2 & 3: Preprocessing & Assembly
**Concept**: The model needs all data in a unified numerical matrix.
**Solution**: We define the routing rules in `ColumnTransformer` and drop it into a `Pipeline`.


In [ ]:
numeric_features = ['age', 'bmi']
categorical_features = ['smoker']

# Step A: Define Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(), categorical_features)
])

# Step B: Define Assembly (Pipeline)
workflow = Pipeline([
    ('pre', preprocessor),
    ('model', LinearRegression())
])

print("Pipeline assembled successfully!")

---
### 1.4 Phase 4 & 5: Training & Evaluation
**Concept**: We train the complex pipeline and evaluate its **R-Squared ($R^2$)**. R-Squared tells us the percentage of variance in the charges that is explained by our features.


In [ ]:
# Train everything in one command
workflow.fit(X_train, y_train)

# Evaluate
y_pred = workflow.predict(X_test)
print(f"R-Squared Accuracy: {r2_score(y_test, y_pred):.2%}")

---
### 1.5 Phase 5: Cross-Validation
**Concept**: Did we just get lucky with our $R^2$ of 74%?
**Solution**: We run a 5-fold cross-validation, using $R^2$ as our scoring metric.


In [ ]:
scores = cross_val_score(workflow, X, y, scoring='r2', cv=5)
print("R-Squared Scores across 5 folds:", np.round(scores, 3))
print(f"\nAverage CV R-Squared: {scores.mean():.2%}")
print(f"Standard Deviation: {scores.std():.2%} (How much the score fluctuates)")

> **Observation**: The cross-validation score shows incredible stability. A standard deviation of ~1% means the model's performance is incredibly reliable regardless of how the data is shuffled.

**Task 1**: In the `numeric_features` list from Phase 2, remove `'bmi'`. Rerun the entire notebook. How much does the Average CV R-Squared drop when the model doesn't know the patient's BMI?

<details>
<summary><strong> Click here for Solution (Try it yourself first!)</strong></summary>

```python
numeric_features = ['age'] # 'bmi' removed
categorical_features = ['smoker']
'''
If you rerun everything, the R^2 score drops to roughly 72%. This tells us that while BMI is important, the smoker feature is doing the vast majority of the heavy lifting. Without smoker (in Lab 1), our model was terrible.
'''
```
</details>

---
### Summary
By adding `smoker` and `bmi` to our `Pipeline` using `ColumnTransformer`, the model accuracy skyrocketed! This proves that selecting professional features and using pipelines for clean routing is the secret to building high-performance AI.